<a href="https://colab.research.google.com/github/arelkeselbri/gsi073/blob/main/Aula02_tokenizacao_pratica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 2 — Tokenização (prática)

Neste laboratório vamos percorrer o **pipeline completo de tokenização** da biblioteca
[`tokenizers`](https://huggingface.co/docs/tokenizers) da Hugging Face:

```
Normalização → Pré-tokenização → Modelo (BPE) → Pós-processamento
```

**Objetivos de aprendizagem**

1. Distinguir os diferentes pré-tokenizadores (`Whitespace`, `Punctuation`, `ByteLevel`, `Metaspace`).
2. Treinar um tokenizador BPE do zero e inspecionar o vocabulário.
3. Montar o pipeline de `encode` com normalização e pós-processamento.
4. Comparar as abordagens *ByteLevel* (GPT-2) e *SentencePiece* (mT5).
5. Avaliar a eficiência de um tokenizador com métricas simples.

> 🔧 Ao longo do notebook há células marcadas com **`# TODO (exercício)`**. Complete-as.
> As células de exercício **não vêm com gabarito**.

## Setup

Instale as dependências (no Colab, execute uma vez).

In [ ]:
# No Colab/ambiente limpo, descomente a linha abaixo:
# !pip install -q tokenizers transformers pandas

import tokenizers
print("tokenizers:", tokenizers.__version__)

## Parte 1 — Pré-tokenização

A **pré-tokenização** quebra o texto em pedaços ("palavras") *antes* de o modelo BPE agir.
Cada estratégia decide de forma diferente onde estão as fronteiras e o que fazer com
espaços e pontuação. O retorno é uma lista de `(pedaço, (início, fim))`, onde o par
`(início, fim)` é o *offset* (posição) do pedaço no texto original.

### 1.1 — Whitespace

In [ ]:
from tokenizers import Tokenizer, models, pre_tokenizers

tok_ws = Tokenizer(models.BPE())
tok_ws.pre_tokenizer = pre_tokenizers.Whitespace()

frase = "Não, será punido o criminoso."
print(tok_ws.pre_tokenizer.pre_tokenize_str(frase))

### 1.2 — Whitespace + Punctuation

In [ ]:
tok_punc = Tokenizer(models.BPE())
tok_punc.pre_tokenizer = pre_tokenizers.Sequence([
    pre_tokenizers.Whitespace(),
    pre_tokenizers.Punctuation(),
])

print(tok_punc.pre_tokenizer.pre_tokenize_str(frase))

### 1.3 — ByteLevel (estilo GPT-2)

Observe o caractere `Ġ`: ele representa um espaço que precede a palavra.

In [ ]:
tok_byte = Tokenizer(models.BPE())
tok_byte.pre_tokenizer = pre_tokenizers.ByteLevel()

print(tok_byte.pre_tokenizer.pre_tokenize_str(frase))

### 1.4 — Metaspace (estilo SentencePiece)

Aqui o espaço vira o caractere `▁` (U+2581).

In [ ]:
tok_meta = Tokenizer(models.BPE())
tok_meta.pre_tokenizer = pre_tokenizers.Metaspace()

print(tok_meta.pre_tokenizer.pre_tokenize_str(frase))

### ✏️ Exercício 1

Use a **mesma frase** nos quatro pré-tokenizadores acima e responda no código:

1. Quantos pedaços cada pré-tokenizador produz? (dica: `len(...)`)
2. Qual deles trata a vírgula e o ponto como pedaços separados?
3. Teste uma frase com número e símbolo, ex.: `"Custou R$ 1.999,90 em 2024!"`,
   e observe as diferenças.

In [ ]:
# TODO (exercício 1)
frase_ex = "Custou R$ 1.999,90 em 2024!"

# 1) Compare a quantidade de pedaços de cada pré-tokenizador:
# for nome, tok in [("ws", tok_ws), ("punc", tok_punc), ("byte", tok_byte), ("meta", tok_meta)]:
#     pedacos = ...
#     print(nome, len(pedacos), pedacos)

# 2) e 3) Escreva suas observações como comentário aqui:


## Parte 2 — Treinamento de um BPE

O algoritmo **BPE (Byte Pair Encoding)**:

1. Começa com todos os caracteres do corpus como tokens.
2. Encontra o par de tokens adjacentes mais frequente e o funde em um novo token.
3. Repete até atingir o tamanho de vocabulário desejado.

> ⚠️ Usaremos um corpus minúsculo só para fins didáticos. Por isso o vocabulário será
> pequeno e, mais adiante, textos novos vão gerar muitos `<unk>` (token desconhecido).

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers

# 1. Corpus de treino (propositalmente pequeno)
corpus = ["Hello, world!", "Hello there", "World of BPE"]
print("Corpus de treino:", corpus, "\n")

# 2. Configuração do tokenizador
tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()

bpe_trainer = trainers.BpeTrainer(
    vocab_size=50,
    special_tokens=["<pad>", "<unk>", "<s>", "</s>"],
)

# 3. Treinamento
tokenizer.train_from_iterator(corpus, trainer=bpe_trainer)
vocab = tokenizer.get_vocab()
print(f"Tamanho do vocabulário: {len(vocab)}\n")

# 4. Visualizando parte do vocabulário (ordenado por id)
sorted_vocab = sorted(vocab.items(), key=lambda kv: kv[1])[:20]
for token, idx in sorted_vocab:
    print(f"{idx:>3} -> {repr(token)}")

# 5. Salvando e recarregando
tokenizer.save("bpe_tokenizer.json")
tokenizer_new = Tokenizer.from_file("bpe_tokenizer.json")

# 6. Testando em novas frases
textos = ["O rato roeu a roupa do rei de Roma", "Hello, world."]
print("\nTokenização de exemplos:")
for texto in textos:
    out = tokenizer_new.encode(texto)
    print(f"Texto: {texto}")
    print(f"Tokens: {out.tokens}")
    print(f"IDs:    {out.ids}\n")

### ✏️ Exercício 2

1. Retreine o tokenizador com `vocab_size = 100` e `vocab_size = 30`.
   Como muda a tokenização de `"Hello, world."`?
2. Acrescente frases novas ao `corpus` (em português) e veja se os `<unk>` diminuem
   na frase do *rato*.

In [ ]:
# TODO (exercício 2)
# Dica: copie o bloco de treino acima, mude vocab_size e/ou corpus, e compare a saída.


## Parte 3 — Pipeline de `encode`

Aqui montamos as quatro etapas: **normalização → pré-tokenização → modelo → pós-processamento**.

In [ ]:
from tokenizers import Tokenizer, normalizers, processors
from tokenizers.normalizers import NFD, StripAccents, Lowercase
from tokenizers.pre_tokenizers import Whitespace, Digits, Sequence
from tokenizers.processors import TemplateProcessing

# Carregar o tokenizador treinado na Parte 2
tokenizer = Tokenizer.from_file("bpe_tokenizer.json")

# ---------- Normalização ----------
normalizer = normalizers.Sequence([
    NFD(),          # decompõe acentos
    Lowercase(),    # tudo minúsculo
    StripAccents(), # remove acentos
])
texto = "Héllò hôw are ü?"
print("Antes: ", texto)
print("Depois:", normalizer.normalize_str(texto), "\n")
tokenizer.normalizer = normalizer

# ---------- Pré-tokenização ----------
pre_tok = Sequence([
    Whitespace(),
    Digits(individual_digits=True),
])
texto2 = "Hello! How are you? Tenho R$ 213,12."
print("Pré-tokenização:", pre_tok.pre_tokenize_str(texto2), "\n")
tokenizer.pre_tokenizer = pre_tok

# ---------- Modelo ----------
# BPE já carregado do arquivo bpe_tokenizer.json

# ---------- Pós-processamento ----------
# OBS: os ids abaixo (1, 2) devem corresponder a tokens que existem no vocabulário.
# Neste corpus didático não há [CLS]/[SEP]; trocamos por <s>/</s>, que foram
# definidos como special_tokens no treino.
tokenizer.post_processor = TemplateProcessing(
    single="<s> $A </s>",
    pair="<s> $A </s> $B:1 </s>:1",
    special_tokens=[
        ("<s>", tokenizer.token_to_id("<s>")),
        ("</s>", tokenizer.token_to_id("</s>")),
    ],
)

# ---------- Aplicando tudo ----------
encoded = tokenizer.encode("olá mundo")
print("Tokens:", encoded.tokens)
print("IDs:   ", encoded.ids)

### ✏️ Exercício 3

1. Remova o `StripAccents` da normalização e tokenize `"olá mundo"` de novo.
   O que muda nos tokens?
2. Troque o `single` do `TemplateProcessing` para incluir só `</s>` no fim.
3. (Reflexão, responda em comentário) Por que o pós-processador precisa que o id
   informado em `special_tokens` exista no vocabulário?

In [ ]:
# TODO (exercício 3)


## Parte 4 — ByteLevel (GPT-2) vs SentencePiece (mT5)

Compararemos dois tokenizadores reais já treinados. Isso baixa modelos da Hugging Face
(precisa de internet).

In [ ]:
from transformers import AutoTokenizer
import unicodedata

BYTELEVEL_MODEL = "openai-community/gpt2"
SENTPIECE_MODEL = "google/mt5-small"

tok_byte = AutoTokenizer.from_pretrained(BYTELEVEL_MODEL)
tok_spm  = AutoTokenizer.from_pretrained(SENTPIECE_MODEL)

# Garantir pad_token para o GPT-2
if tok_byte.pad_token is None and tok_byte.eos_token is not None:
    tok_byte.pad_token = tok_byte.eos_token

text = "Vamos comer, vovó! 🙂"
print(f"Texto: {text}\n")

def encode_details(tokenizer, name):
    enc = tokenizer(text, add_special_tokens=True)
    ids = enc["input_ids"]
    tokens = tokenizer.convert_ids_to_tokens(ids)
    print(f"=== {name} ===")
    print("Tokens:    ", tokens)
    print("IDs:       ", ids)
    print("Qtd tokens:", len(tokens))
    print("Decoded:   ", tokenizer.decode(ids))
    print()

encode_details(tok_byte, "ByteLevel (GPT-2)")
encode_details(tok_spm,  "SentencePiece (mT5)")

# Inspeção Unicode do texto
print("Caracteres Unicode do texto:")
for ch in text:
    print(f"{repr(ch)} -> {unicodedata.name(ch, 'UNKNOWN')}")

### ✏️ Exercício 4

1. Qual dos dois produz **menos** tokens para a frase de exemplo?
2. Como cada um representa o emoji `🙂`? E o acento de `vovó`?
3. Teste uma frase só em inglês e outra só em português. Algum dos dois fica
   sistematicamente mais "econômico" em uma das línguas?

In [ ]:
# TODO (exercício 4)


## Parte 5 — Avaliação de eficiência

Métricas simples para comparar tokenizadores:

- **TPC** (tokens por caractere) e **TPW** (tokens por palavra): quanto menor, mais compacto.
- **% de `<unk>`**: cobertura do vocabulário (quanto menor, melhor).
- **Reversibilidade**: `decode(encode(x)) == x`?

> ⚠️ **Atenção pedagógica:** o tokenizador da Parte 2 foi treinado num corpus mínimo
> e usa o pré-tokenizador `Whitespace`, que **não** tem um *decoder* configurado para
> reconstruir o texto. Por isso, espere `% de <unk>` alto e reversibilidade baixa aqui.
> Isso é justamente o que o exercício pede para você observar e explicar.

In [ ]:
import pandas as pd
from tokenizers import Tokenizer

tokenizer = Tokenizer.from_file("bpe_tokenizer.json")

test_texts = [
    "O rato roeu a roupa do rei de Roma.",
    "Aprender tokenização é divertido!",
    "GPT-2 e mT5 usam abordagens diferentes.",
    "Python é ótimo para NLP 😄",
]

def evaluate_tokenizer(tokenizer, texts):
    stats = []
    for t in texts:
        enc = tokenizer.encode(t)
        stats.append({
            "text": t,
            "chars": len(t),
            "words": len(t.split()),
            "tokens": len(enc.tokens),
            "unk": enc.tokens.count("<unk>"),
            "decoded_ok": tokenizer.decode(enc.ids) == t,
        })
    return stats

df = pd.DataFrame(evaluate_tokenizer(tokenizer, test_texts))

tpc        = (df["tokens"] / df["chars"]).mean()
tpw        = (df["tokens"] / df["words"]).mean()
unk_rate   = df["unk"].sum() / df["tokens"].sum() * 100
decode_acc = df["decoded_ok"].mean() * 100

print("=== Métricas de eficiência ===")
print(f"Tokens por caractere (TPC):              {tpc:.3f}")
print(f"Tokens por palavra (TPW):                {tpw:.3f}")
print(f"Percentual de <unk>:                     {unk_rate:.2f}%")
print(f"Reversibilidade (decode == original):    {decode_acc:.1f}%")
print(f"Tamanho médio da sequência:              {df['tokens'].mean():.1f} tokens/frase")

df

### ✏️ Exercício 5

1. Rode as mesmas métricas para o **GPT-2** e o **mT5** da Parte 4
   (use `tok_byte` / `tok_spm`). Atenção: a API deles é diferente
   (`tokenizer(t)["input_ids"]`, `tokenizer.decode(...)`).
2. Monte uma tabela comparando TPC, TPW e reversibilidade dos três tokenizadores.
3. (Reflexão) Por que o tokenizador da Parte 2 tem reversibilidade baixa, enquanto
   GPT-2/mT5 reconstroem o texto quase perfeitamente?

In [ ]:
# TODO (exercício 5)
